# Rossmann Store Sales — Pipeline Walkthrough

> Daily store-sales forecasting (time-series regression).

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH, low_memory=False)
print(df.shape)
df.head()

(15000, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,239,6,2014-08-02,5203,463,1,0,0,0,d,c,610.0,NaN,NaN,0,NaN,NaN,NaN
1,67,4,2013-05-16,8590,867,1,1,0,0,a,c,410.0,2.0,2006.0,0,NaN,NaN,NaN
2,626,1,2015-04-20,6465,621,1,0,0,0,c,c,10740.0,11.0,2013.0,0,NaN,NaN,NaN
3,618,4,2015-07-23,7250,568,1,0,0,0,d,c,9910.0,NaN,NaN,0,NaN,NaN,NaN
4,185,1,2014-02-24,4339,399,1,0,0,0,d,c,1860.0,5.0,2015.0,0,NaN,NaN,NaN


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
X, y, dates = Preprocessor().run(df)
print("open-day records:", X.shape, "| date range:", dates.min().date(), "->", dates.max().date())
X.head()

open-day records: (14999, 16) | date range: 2013-01-02 -> 2015-07-31


,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,504,3,2013-01-02,1,0,0,1,c,c,820.0,NaN,NaN,0,NaN,NaN,NaN
1,30,3,2013-01-02,1,0,0,1,a,a,40.0,2.0,2014.0,1,10.0,2014.0,"Mar,Jun,Sept,Dec"
2,264,3,2013-01-02,1,0,0,1,a,a,180.0,3.0,2014.0,0,NaN,NaN,NaN
3,1021,3,2013-01-02,1,0,0,1,a,a,1080.0,5.0,2011.0,0,NaN,NaN,NaN
4,735,3,2013-01-02,1,0,0,1,d,c,1920.0,4.0,2005.0,0,NaN,NaN,NaN


## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "trained_at": "2026-06-14T22:19:52+00:00",
  "dataset": "pratyushakar/rossmann-store-sales",
  "data_sha256": "unknown",
  "target": "Sales",
  "problem": "daily store sales forecast (time-series regression)",
  "best_model": "HistGradientBoostingRegressor",
  "best_params": {
    "model__max_leaf_nodes": 127,
    "model__max_iter": 800,
    "model__max_depth": null,
    "model__learning_rate": 0.12073082874598748,
    "model__l2_regularization": 1.0
  },
  "baseline": {
    "model": "LinearRegression",
    "rmse": 2755.3,
    "mae": 1989.4,
    "r2": 0.2225,
    "rmspe": 0.4613
  },
  "holdout": {
    "rmse": 1620.29,
    "mae": 1052.97,
    "r2": 0.7311,
    "rmspe": 0.2471,
    "slice_rmspe": {
      "StoreType=a": 0.2882,
      "StoreType=b": 0.1822,
      "StoreType=c": 0.1937,
      "StoreType=d": 0.186,
      "Promo=0": 0.265,
      "Promo=1": 0.2241
    },
    "n_holdout": 126651
  },
  "business": {
    "headline": "Daily store sales are predicte

## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
store = {"Store":1,"DayOfWeek":4,"Date":"2015-08-01","Open":1,"Promo":1,"StateHoliday":"0","SchoolHoliday":1,"StoreType":"c","Assortment":"a","CompetitionDistance":1270,"CompetitionOpenSinceMonth":9,"CompetitionOpenSinceYear":2008,"Promo2":0,"Promo2SinceWeek":None,"Promo2SinceYear":None,"PromoInterval":""}
print("predicted daily sales:", round(pred.predict_one(store), 2))

predicted daily sales: 5858.61


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```